# Lilly — train the listening on real Bosnian speech

Set two things in the panel on the right:

- **Session options → Accelerator → GPU T4 x2** (or P100)
- **Session options → Internet → On**

No dataset upload needed — this notebook fetches its own speech data.

Then **Save Version → Save & Run All (Commit)** and close the tab.

It measures before and after, on clips it never trains on. That comparison is
the whole point: a run that does not lower the error rate did not work, however
cleanly it finished.


In [ ]:
# 1. Check we actually got a GPU
!nvidia-smi -L

In [ ]:
# 2. Get the Lilly code
%cd /kaggle/working
!rm -rf Lilly && git clone -q https://github.com/ssaaffaakk/Lilly.git
%cd /kaggle/working/Lilly

In [ ]:
# 3. Install what we need (~3 min)
!pip -q install transformers==4.49.0 accelerate==1.3.0 faster-whisper ctranslate2 \
    soundfile scipy pyarrow

In [ ]:
# 4. Download the speech: clips to train on, and clips held back to judge with
!python3 data/scripts/download_speech_data.py

In [ ]:
# 5. BEFORE: how bad is the untrained listener on Bosnian it never heard?
!python3 training/train_speech.py --base openai/whisper-small \
    --convert-only /kaggle/working/listen-before
!python3 training/evaluate_speech.py --data data/speech/test.tsv \
    --model /kaggle/working/listen-before --limit 200 --show 3

In [ ]:
# 6. THE REAL TRAINING (~1-2 hours)
!python3 training/train_speech.py --data data/speech/train.tsv

In [ ]:
# 7. AFTER: the same clips, the same measure. Lower is better.
!python3 training/evaluate_speech.py --data data/speech/test.tsv \
    --limit 200 --show 3

In [ ]:
# 8. Package the trained listener so it survives the run
!cd /kaggle/working/Lilly && zip -qr /kaggle/working/lilly-listen.zip models/lilly/listen
!ls -lh /kaggle/working/lilly-listen.zip

**Done.** Compare the two error rates printed by cells 5 and 7 — same clips,
same measure, so the difference is real.

If it dropped, download `lilly-listen.zip` from the Output tab and unzip it over
`models/lilly/listen/`. If it did not drop, keep the listener you have: more
data, or more epochs, before another run.
